# ontoagents: usage walkthrough

**Requirements:**
`uv sync --group notebooks`, and a kernel from the project's `.venv`. Java on `PATH` is needed only for HermiT; without it the relevant cells report that the check is skipped. This notebook **never calls a real model**, so it costs nothing and needs no keys. Numbers produced by the fake LLM are meaningless; the point is the flow.

**Outline:**
1. Setup
2. Data and competency questions
3. Ontology draft and `Signature`
4. Validators
5. LLM cache
6. An offline run through the CLI
7. The loop from the inside: repair and budget
8. Agent system vs Ontogenia
9. A real benchmark
10. Exercise, pitfalls, extensions

## 1. Setup

In [1]:
import json
import os
import subprocess
import sys
import tempfile
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)

# Results of this notebook go to a temp dir, not to experiments/results.
RESULTS = Path(tempfile.mkdtemp(prefix="ontoagents-demo-"))


def cli(*args) -> int:
    """Run `ontoagents` with the same interpreter as the kernel."""
    proc = subprocess.run(
        [sys.executable, "-m", "ontoagents.cli", *map(str, args)],
        capture_output=True,
        text=True,
    )
    print((proc.stdout + proc.stderr).rstrip())
    return proc.returncode


print("repo:   ", ROOT)
print("results:", RESULTS)

repo:    /Users/daniel/Developer/projects/onto-agents
results: /var/folders/2h/3n_fb0z505ncl73c6_jrtw9r0000gn/T/ontoagents-demo-d4d7ap7b


## 2. Data and competency questions

`ontoagents catalog` prints the inventory: number of CQs, whether a reference ontology exists, classes and properties (own / imported) and gold SPARQL queries. SILKNOW has no reference (`ref = no`).

In [2]:
! uv run ontoagents catalog

ontology       CQs  ref  cls own/imp  props own/imp  gold rq
DOREMUS         58  yes       83/132        279/423       53
DemCare        107  yes       248/12         113/41       60
Odeuropa        74  yes         16/6           10/0       74
Polifonia      361  yes        210/8         279/20        0
SILKNOW         78   no            -              -        -
awo             14  yes         30/0            5/0        7
noria-o         26  yes         54/2          135/0       25
ontodt          14  yes        401/4           13/3       13
stuff           11  yes         78/0           37/0        9
swo             88  yes    1027/3039          36/20       42


The loader reads only what the loop may see: `id`, `ontology`, `question`, `group`. The `rq` field (gold queries) is dropped right here. `GroupPlanner` splits the CQs into batches of 5 without mixing groups.

In [3]:
from ontoagents.agents.planner import GroupPlanner
from ontoagents.data.loader import YamlCQLoader

cqs = YamlCQLoader(ROOT / "data").load("awo")
print(f"{len(cqs)} CQs in awo\n")
for cq in cqs[:4]:
    print(f"{cq.id:7s} placeholder={cq.has_placeholders!s:5s} {cq.question}")

print()
for batch in GroupPlanner(batch_size=5).plan(cqs).batches:
    print(batch.id, batch.cq_ids)

14 CQs in awo

awo_1   placeholder=False Which animal eats which other animal?
awo_2   placeholder=True  Is [this animal] a herbivore?
awo_3   placeholder=True  Which plant parts does [this omnivorous or herbivorourus animal] eat?
awo_4   placeholder=False Does a lion eat plants or plant parts?

b001 ('awo_1', 'awo_2', 'awo_3', 'awo_4', 'awo_5')
b002 ('awo_6', 'awo_7', 'awo_8', 'awo_9', 'awo_10')
b003 ('awo_11', 'awo_12', 'awo_13', 'awo_14')


**The leakage boundary in action.** `CompetencyQuestion` has `extra="forbid"` and no field for SPARQL, so trying to smuggle a gold query in fails.

In [4]:
from pydantic import ValidationError

from ontoagents.domain import CompetencyQuestion

try:
    CompetencyQuestion(id="x", ontology="awo", question="Which animal eats what?", sparql="SELECT *")
except ValidationError as exc:
    err = exc.errors()[0]
    print("rejected:", err["type"], err["loc"])

rejected: extra_forbidden ('sparql',)


## 3. Ontology draft and `Signature`

`OntologyDraft` is Turtle text plus a `base_iri`. `merge` does not mutate: it returns a **new** draft (set union of triples). This is why the agents' fragments are *add-only*. `Signature` is the ontology's vocabulary, the only view of it that the Generator, Translator and Critic get.

In [5]:
from ontoagents.domain import OntologyDraft

BASE = "http://ontoagents.example.org/demo#"

fragment = '''
:Animal a owl:Class ; rdfs:label "animal" ; rdfs:comment "Any animal." .
:Plant  a owl:Class ; rdfs:label "plant"  ; rdfs:comment "Any plant." .
:eats a owl:ObjectProperty ; rdfs:label "eats" ; rdfs:comment "An animal eats something." ;
      rdfs:domain :Animal ; rdfs:range :Animal .
'''

draft = OntologyDraft.empty(BASE)
candidate = draft.merge(fragment)       # draft itself is unchanged

print("triples:", len(draft), "->", len(candidate))
print(draft.diff(candidate))
print()
print(candidate.signature().to_prompt(BASE))

triples: 1 -> 12
added_classes=('http://ontoagents.example.org/demo#Animal', 'http://ontoagents.example.org/demo#Plant') removed_classes=() added_properties=('http://ontoagents.example.org/demo#eats',) removed_properties=() added_triples=11 removed_triples=0

Classes:
  :Animal "animal"
  :Plant "plant"
Properties:
  :eats [object] "eats" domain=:Animal range=:Animal


## 4. Validators

**Structural SPARQL check.** Queries are never executed (a TBox has no data). The checker verifies that classes and properties exist, are used in the right role, and fit the types of the variables.

In [6]:
from ontoagents.validation.cq_coverage import StructuralQueryChecker

checker = StructuralQueryChecker()
sig = candidate.signature()

queries = {
    "good": "SELECT ?a ?b WHERE { ?a a :Animal ; :eats ?b . ?b a :Animal . }",
    "missing-terms": "SELECT ?a WHERE { ?a a :Animal ; :drinks ?w . ?w a :Water . }",
    "wrong-role": "SELECT ?a WHERE { ?a a :eats . }",
}
for name, sparql in queries.items():
    check = checker.check(name, sparql, sig, BASE)
    print(f"{name:14s} {check.status.value:5s}", *[f"\n    - {i.message}" for i in check.issues])

good           pass 
missing-terms  fail  
    - property http://ontoagents.example.org/demo#drinks is not defined in the ontology 
    - class http://ontoagents.example.org/demo#Water is not defined in the ontology
wrong-role     fail  
    - http://ontoagents.example.org/demo#eats is a property but is used as a class


**The pipeline.** Validators run cheapest first and stop at the first failure: `syntax` → `consistency` (HermiT) → `cq_coverage` (→ `regression`). `to_feedback()` is exactly the text the Critic receives.

In [7]:
from ontoagents.validation.base import ValidationContext
from ontoagents.validation.consistency import ConsistencyValidator, java_available
from ontoagents.validation.cq_coverage import CQCoverageValidator
from ontoagents.validation.pipeline import ValidatorPipeline
from ontoagents.validation.syntax import SyntaxValidator

print("Java available:", java_available())
pipeline = ValidatorPipeline([SyntaxValidator(), ConsistencyValidator(), CQCoverageValidator()])
context = ValidationContext(
    base_iri=BASE,
    candidate=candidate,
    fragment=fragment,
    batch_sparql={"good": queries["good"], "missing-terms": queries["missing-terms"]},
)
report = pipeline.run(context)
print("green:", report.passed, "| first failure:", report.failed_validator, "\n")
print(report.to_feedback())

Java available: True
green: False | first failure: cq_coverage 

[syntax] PASSED
[consistency] PASSED
[cq_coverage] FAILED
  - error (CQ missing-terms): property http://ontoagents.example.org/demo#drinks is not defined in the ontology
  - error (CQ missing-terms): class http://ontoagents.example.org/demo#Water is not defined in the ontology
  query for CQ missing-terms:
      SELECT ?a WHERE { ?a a :Animal ; :drinks ?w . ?w a :Water . }


**HermiT in action.** `Lion` is a subclass of two disjoint classes, so it is unsatisfiable. Without Java the consistency validator is skipped (`skipped=True`) and **the run still looks green**, so always check `java_available()`.

In [8]:
broken = candidate.merge('''
:Herbivore a owl:Class .
:Carnivore a owl:Class ; owl:disjointWith :Herbivore .
:Lion a owl:Class ; rdfs:subClassOf :Herbivore, :Carnivore .
''')
report = ValidatorPipeline([SyntaxValidator(), ConsistencyValidator()]).run(
    ValidationContext(base_iri=BASE, candidate=broken)
)
for r in report.reports:
    print(f"{r.validator:12s} passed={r.passed} skipped={r.skipped}")
    for issue in r.issues:
        print("   ", issue.severity, issue.message)

syntax       passed=True skipped=False
consistency  passed=False skipped=False
    error class http://ontoagents.example.org/demo#Lion is unsatisfiable (equivalent to owl:Nothing)


## 5. LLM cache

`CachingLLMClient` decorates any client. The key is a hash of (namespace, model, messages, temperature, max_tokens). A hit returns `cached=True` with the **original** token counts, so cost metrics stay truthful. In the project the cache lives in `.cache/llm` (git-ignored).

In [9]:
from ontoagents.llm.cache import CachingLLMClient
from ontoagents.llm.client import user
from ontoagents.llm.fake import FakeLLMClient

inner = FakeLLMClient(replies=["model reply"])
cached = CachingLLMClient(inner, Path(tempfile.mkdtemp()), namespace="seed=42")

first = cached.complete([user("Hi")])
second = cached.complete([user("Hi")])
print("cached:", first.cached, "->", second.cached)
print("model calls:", len(inner.calls))
print("tokens preserved:", first.input_tokens == second.input_tokens)

cached: False -> True
model calls: 1
tokens preserved: True


## 6. An offline run through the CLI

`dry-run.yaml` uses `provider: fake` (`HeuristicLLMClient`: rules instead of a model). It exercises the whole pipeline: planning, generation, translation, validation, evaluation and the report.

In [11]:
cli("run", "--config", "experiments/configs/dry-run.yaml", "--results-dir", RESULTS)
# Equivalent to:
# ! uv run ontoagents run --config experiments/configs/dry-run.yaml --results-dir ...

[ontoagents] experiment=dry-run system=agents-group llm=fake:fake-model ontologies=['awo', 'stuff', 'SILKNOW']
  awo        class_f1=0.255 property_f1=0.235 posthoc_covered_rate=1.000 llm_calls=6
  stuff      class_f1=0.103 property_f1=0.044 posthoc_covered_rate=1.000 llm_calls=6
  SILKNOW    class_f1=None property_f1=None posthoc_covered_rate=1.000 llm_calls=46


0

Result layout: `<ontology>.json` (config, metrics, CQ statuses, committed SPARQL, alignment, trace), `<ontology>.ttl` (the ontology) and `REPORT.md`.

In [12]:
run_dir = RESULTS / "dry-run"
print(sorted(p.name for p in run_dir.iterdir()))

result = json.loads((run_dir / "awo.json").read_text())
print("\nsystem:", result["system"], "| stop_reason:", result["stop_reason"], "| failed CQs:", result["failed_cqs"])
print("events:", dict(Counter(e["kind"] for e in result["trace"])))

metrics = result["metrics"]
for key in ("class_n_generated", "class_n_reference", "class_n_matched", "class_precision",
            "class_recall", "class_recall_cq_relevant", "internal_covered_rate",
            "posthoc_covered_rate", "llm_calls", "repairs", "rollbacks"):
    print(f"  {key:26s} {metrics.get(key)}")

print("\nclass alignment (first 5):")
for m in result["alignment"]["class"][:5]:
    print(f"  {m['score']:.2f}  {m['generated'].rsplit('#', 1)[-1]:12s} ~ {m['reference'].rsplit('#', 1)[-1]}")

['REPORT.md', 'SILKNOW.json', 'SILKNOW.ttl', 'awo.json', 'awo.ttl', 'stuff.json', 'stuff.ttl']

system: agents-group | stop_reason: None | failed CQs: []
events: {'plan': 1, 'generate': 3, 'translate': 3, 'validate': 3, 'commit': 3}
  class_n_generated          17
  class_n_reference          30
  class_n_matched            6
  class_precision            0.35294117647058826
  class_recall               0.2
  class_recall_cq_relevant   0.8333333333333334
  internal_covered_rate      1.0
  posthoc_covered_rate       1.0
  llm_calls                  6
  repairs                    0
  rollbacks                  0

class alignment (first 5):
  1.00  Animal       ~ animal
  1.00  Carnivore    ~ carnivore
  1.00  Habitat      ~ Habitat
  1.00  Herbivore    ~ herbivore
  1.00  Lion         ~ lion


## 7. The loop from the inside: repair and budget

The fake LLM passes everything on the first try, so `dry-run` **never exercises repair**. Here we swap in a client whose Generator returns a useless draft on its first attempt. The Translator writes queries against classes that do not exist, `cq_coverage` goes red, the Critic writes instructions, and the second attempt passes.

In [13]:
from ontoagents.cli import build_generator
from ontoagents.config import ExperimentConfig, LLMConfig, SystemConfig
from ontoagents.domain import GenerationInputs
from ontoagents.llm.client import LLMResponse
from ontoagents.llm.heuristic import HeuristicLLMClient


class FlakyGenerator(HeuristicLLMClient):
    """The Generator's first attempt is useless; after feedback it behaves normally."""

    def complete(self, messages, *, temperature=None, max_tokens=None):
        system_text = " ".join(m.content for m in messages if m.role == "system")
        user_text = "\n".join(m.content for m in messages if m.role == "user")
        if "builds OWL ontologies incrementally" in system_text and "rejected by the validators" not in user_text:
            return LLMResponse(text="```turtle\n:Placeholder a owl:Class .\n```", model="flaky")
        return super().complete(messages, temperature=temperature, max_tokens=max_tokens)


def run_system(llm, system_cfg, n_cqs=5):
    cfg = ExperimentConfig(name="demo", llm=LLMConfig(provider="fake"), system=system_cfg)
    inputs = GenerationInputs(ontology="awo", base_iri=BASE, cqs=tuple(cqs[:n_cqs]))
    return build_generator(cfg, llm).generate(inputs)


repaired = run_system(FlakyGenerator(), SystemConfig(max_repair=3))
for event in repaired.trace:
    if event.kind in ("generate", "validate", "repair", "commit", "rollback"):
        d = event.data
        extra = {k: d[k] for k in ("attempt", "passed", "failed_validator") if k in d}
        print(f"{event.kind:9s}", extra)
print("\nstop_reason:", repaired.stop_reason, "| failed CQs:", repaired.failed_cqs)

generate  {'attempt': 0}
validate  {'attempt': 0, 'passed': False, 'failed_validator': 'cq_coverage'}
repair    {}
generate  {'attempt': 1}
validate  {'attempt': 1, 'passed': True, 'failed_validator': None}
commit    {}

stop_reason: None | failed CQs: ()


**Budget.** `max_repair` applies per batch (when exceeded: rollback). `max_llm_calls`, `max_tokens` and `max_iterations` are **global** (when exceeded: `stop`). There are no global limits by default. Here a limit of 2 calls stops the run after the first batch, and the second batch (5 CQs) is reported as failed.

In [14]:
limited = run_system(HeuristicLLMClient(), SystemConfig(max_llm_calls=2), n_cqs=10)
print("stop_reason:", limited.stop_reason)
print("committed: ", sorted(k for k, v in limited.cq_status.items() if v.value != "fail"))
print("failed:    ", list(limited.failed_cqs))

stop_reason: budget: llm_calls
committed:  ['awo_1', 'awo_2', 'awo_3', 'awo_4', 'awo_5']
failed:     ['awo_6', 'awo_7', 'awo_8', 'awo_9', 'awo_10']


## 8. Agent system vs Ontogenia

Both systems implement the same `OntologyGenerator` interface and receive identical inputs, so they differ only by config file (or the `--system` flag). Here both run offline on `awo` with 10 CQs. The real comparison is the same commands with `--provider claude-code` or `anthropic`.

In [15]:
common = ["--provider", "fake", "--ontology", "awo", "--max-cqs", 10, "--results-dir", RESULTS]
cli("run", "--config", "experiments/configs/agents-full.yaml", "--name", "demo-agents", *common)
cli("run", "--config", "experiments/configs/baseline-ontogenia.yaml", "--name", "demo-ontogenia", *common)

[ontoagents] experiment=demo-agents system=agents-group llm=fake:claude-sonnet-5-5 ontologies=['awo']
  awo        class_f1=0.182 property_f1=0.267 posthoc_covered_rate=1.000 llm_calls=4
[ontoagents] experiment=demo-ontogenia system=ontogenia llm=fake:claude-sonnet-5-5 ontologies=['awo']
  awo        class_f1=0.121 property_f1=0.000 posthoc_covered_rate=0.100 llm_calls=10


0

In [16]:
from ontoagents.evaluation.report import load_results, render_markdown

demo = [r for r in load_results(RESULTS) if r["experiment"].startswith("demo-")]
print(render_markdown(demo))

# Results

## demo-agents (`agents-group`)

| Ontology | Class F1 | Class R (CQ-rel.) | Prop. F1 | Prop. R (CQ-rel.) | SubClass agr. | CQ coverage | Gold SPARQL | Consistent | #Cls | #Props | Calls | Tokens |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| awo | 0.182 | 0.500 | 0.267 | 1.000 | – | 1.000 | 0.857 | 1 | 14 | 10 | 4 | 3,243 |

## demo-ontogenia (`ontogenia`)

| Ontology | Class F1 | Class R (CQ-rel.) | Prop. F1 | Prop. R (CQ-rel.) | SubClass agr. | CQ coverage | Gold SPARQL | Consistent | #Cls | #Props | Calls | Tokens |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| awo | 0.121 | 0.167 | 0.000 | 0.000 | – | 0.100 | 0.143 | 1 | 3 | 1 | 10 | 6,682 |

## Macro averages

| Experiment | Class F1 | Class R (CQ-rel.) | Prop. F1 | Prop. R (CQ-rel.) | SubClass agr. | CQ coverage | Gold SPARQL | Consistent | #Cls | #Props | Calls | Tokens |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| demo-agents | 0.182 | 0.500 | 0.267 | 1.000 | – | 1.000 | 0.857 | 1.00

The differences in this table are an artefact of the fake LLM (for example, in Ontogenia the `HeuristicLLMClient` returns, at every step, an ontology containing only the current CQ, which then replaces the previous one), so nothing follows from them.

How to read such a table with real numbers:
- **Independent of the loop:** `Class F1`, `Class R (CQ-rel.)`, `Prop. F1`, `SubClass agr.`, `Gold SPARQL`. These carry the main claim.
- **Dependent on the loop's validators:** `CQ coverage` (`posthoc_*`). The agent system is optimised against the same translator and checker, so it has a built-in advantage here. Treat it as a supplementary metric.
- **Cost:** `Calls` and `Tokens`. Ontogenia makes 1 call per CQ with a growing context; the agents make about 2–4 calls per batch of 5 CQs.

## 9. A real benchmark

The blocks below are **descriptions only** (code blocks, not executed) because they need a model and cost money.

**A small trial run** (start here; Claude Code CLI, no API key):

```bash
uv run ontoagents run --config experiments/configs/agents-full.yaml --ontology awo --max-cqs 14
uv run ontoagents run --config experiments/configs/baseline-ontogenia.yaml --ontology awo --max-cqs 14
uv run ontoagents report
```

**With an API through LangChain:**

```bash
export ANTHROPIC_API_KEY=...
uv run ontoagents run --config experiments/configs/agents-full.yaml --provider anthropic --ontology awo
```

**The full benchmark** (baselines, the system, every ablation, then the report). It is expensive: `agents-full` makes 2–4 calls per batch of 5 CQs, and Polifonia alone has 361 CQs.

```bash
WORKERS=4 experiments/run_all.sh --max-cqs 30
```

**A global budget** (none by default) goes in the `system:` section of the YAML:

```yaml
system:
  kind: agents
  max_repair: 3
  max_llm_calls: 400
  max_tokens: 2000000
```

**Ablations** live in `experiments/configs/`: `ablation-no-critic`, `-llm-planner`, `-no-regression`, `-no-translator`, `-no-repair`. Each changes only the `system:` section.

**Practical notes:**
- Results land in `experiments/results/<experiment>/` (the directory is not git-ignored; add `--results-dir` for trials).
- Model replies are cached in `.cache/llm`, so reruns and new metrics cost nothing. A changed prompt simply stops matching its old cache entries, but a code change that leaves the prompt identical invalidates nothing.
- `provider: claude-code` ignores `temperature` and `seed`; repeatability comes only from the cache.
- `--workers N` runs ontologies in parallel in separate processes.

## 10. Exercise, pitfalls, extensions

**Exercise.** The metrics include a sensitivity curve over the alignment threshold (`class_f1@0.7`, `@0.8`, `@0.85`, `@0.9`, `@1`). Print the class F1 for each threshold for the `awo` run from section 6 and find the threshold step where F1 drops the most. A skeleton to complete follows.

In [17]:
# TODO: collect the `class_f1@<threshold>` keys from `metrics` and print them in ascending order.
sensitivity = {}
print(sensitivity)

{}


**Solution:**

In [18]:
sensitivity = {
    float(key.split("@")[1]): value
    for key, value in metrics.items()
    if key.startswith("class_f1@")
}
thresholds = sorted(sensitivity)
for t in thresholds:
    print(f"threshold {t:<5g} F1 = {sensitivity[t]:.3f}")

drops = {b: sensitivity[a] - sensitivity[b] for a, b in zip(thresholds, thresholds[1:])}
worst = max(drops, key=drops.get)
print(f"\nlargest drop at threshold {worst:g} ({drops[worst]:.3f})")

threshold 0.7   F1 = 0.298
threshold 0.8   F1 = 0.298
threshold 0.85  F1 = 0.255
threshold 0.9   F1 = 0.255
threshold 1     F1 = 0.255

largest drop at threshold 0.85 (0.043)
